# EVA de novo biophysical characterization

Rigorous comparison of **105** yield-gated EVA candidates (**31 pilot** + **74 stream**) against natural Rfam thermoswitches and RefSeq 5′ UTR negatives.

Yield triage that admitted these sequences is still $Z \le -2 \land \Delta P_{\mathrm{RBS}} > 0 \land E_{\mathrm{Rfam}} > 10^{-3}$. This notebook asks whether they also show **cooperative biophysical switching** (Hill sigmoid, heat-shock $T_m$, locked 37 °C baseline) rather than Vienna-specific heuristics.

Marimo counterpart: [`08_eva_denovo_biophysical_characterization.py`](08_eva_denovo_biophysical_characterization.py). Typed helpers live in `src/thermo_sim/eva_denovo_characterization.py`.

PrfA is **not** in the 4-sequence prototype panel; **FourU** is the gold-standard overlay.

Set `RUN_EVA_SWEEPS = True` to fold all 105 passers at 1 °C from 30–60 °C (cached). Default is cache-first so the notebook opens without a multi-engine re-run.


In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from thermo_sim.eva_denovo_characterization import (
    SCORE_WEIGHTS,
    backfill_eva_p_open,
    characterization_paths,
    checklist_by_cohort,
    cohort_stat_tests,
    engine_concordance,
    export_leads,
    flag_software_fragile,
    hill_fits_from_sweeps,
    load_control_panel,
    load_eva_passers,
    load_prototype_overlay,
    merge_eva_fits,
    plot_engine_parity,
    plot_leakiness,
    plot_nh_vs_tm,
    plot_novelty_vs_nh,
    plot_ribbon_curves,
    plot_violins,
    rank_leads,
    ribbon_from_hill_params,
    ribbon_from_sweep_table,
    run_or_load_eva_sweeps,
    save_figure,
    sweep_temps,
    write_checklist_summary,
)

plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight"})

# Flip to True to compute Vienna P_open(T) for the 105 EVA passers (writes cache).
RUN_EVA_SWEEPS = False
BACKFILL_P_OPEN = False

paths = characterization_paths(PROJECT_ROOT)
print("project root:", PROJECT_ROOT)
print("fused:", paths["fused"].exists(), paths["fused"])
print("pilot FASTA:", paths["pilot_fasta"].exists())
print("stream FASTA:", paths["stream_fasta"].exists())

## §1 Cohort ingestion & batch simulation pipeline

Load candidate pools:

- `data/processed/eva_pilot/top_candidates.fasta` ($N=31$)
- `data/processed/eva_stream/top_candidates.fasta` ($N=74$)
- Baseline controls from `fused_features_refseq_dynamic.csv` ($N=1{,}198$ Rfam, $N=1{,}198$ RefSeq)

Temperature sweeps ($30^\circ\mathrm{C}$–$60^\circ\mathrm{C}$, $1^\circ\mathrm{C}$) use the same SD-window unpaired series as `extract_vienna_features`. The 2,396-row panel is **not** re-folded here: ribbons are reconstructed from stored Hill parameters.


In [ ]:
eva = load_eva_passers(paths)
if BACKFILL_P_OPEN:
    eva = backfill_eva_p_open(eva, run=True)
controls = load_control_panel(paths["fused"])
prototypes = load_prototype_overlay(paths)

n_pilot = int((eva["cohort"] == "EVA Pilot Top Passers").sum()) if not eva.empty else 0
n_stream = (
    int((eva["cohort"] == "EVA Stream Top Passers").sum()) if not eva.empty else 0
)
print(f"EVA passers: {len(eva)} (pilot={n_pilot}, stream={n_stream})")
print(f"Controls: {len(controls)}")
print(f"Prototype overlays: {len(prototypes)}")
if not eva.empty:
    display(
        eva[
            [
                "record_id",
                "cohort",
                "seq_length",
                "viennarna_mfe_zscore",
                "viennarna_delta_P_RBS",
                "E_Rfam",
            ]
        ].head()
    )

## §2 Parametric Hill sigmoid fitting

$$\theta(T) = \theta_{\mathrm{min}} + (\theta_{\mathrm{max}} - \theta_{\mathrm{min}}) \frac{T^{n_{\mathrm{H}}}}{T_{\mathrm{m}}^{n_{\mathrm{H}}} + T^{n_{\mathrm{H}}}}$$

Extract $n_{\mathrm{H}}$, $T_{\mathrm{m}}$, $\Delta\theta$, $\theta_{\mathrm{min}}$. Flag $R^2 < 0.95$.


In [ ]:
sweeps = run_or_load_eva_sweeps(
    eva, cache_path=paths["sweep_cache"], run=RUN_EVA_SWEEPS
)
fits = hill_fits_from_sweeps(sweeps)
eva_fitted = merge_eva_fits(eva, fits)
if not fits.empty:
    paths["hill_cache"].parent.mkdir(parents=True, exist_ok=True)
    fits.to_csv(paths["hill_cache"], index=False)
    n_ok = int(fits["fit_converged"].sum()) if "fit_converged" in fits.columns else 0
    print(
        f"Sweep rows={len(sweeps)}; fits={fits['record_id'].nunique()}; R^2>=0.95: {n_ok}"
    )
    display(fits.head())
else:
    print(
        "No EVA temperature cache. Set RUN_EVA_SWEEPS = True to fold the 105 passers."
    )

## §3 Population distribution diagnostics

Statistical comparisons (Mann–Whitney $U$, Kolmogorov–Smirnov):

- EVA Stream vs Natural Rfam Positives
- EVA Stream vs RefSeq Negatives
- EVA Stream vs EVA Pilot

Visuals: four-panel violins ($n_{\mathrm{H}}$, $T_m$, $\Delta P_{\mathrm{RBS}}$, $Z$) and grand-mean temperature-sweep ribbons.


In [ ]:
panel = pd.concat(
    [df for df in (controls, eva_fitted) if df is not None and not df.empty],
    ignore_index=True,
    sort=False,
)
temps = np.asarray(sweep_temps(), dtype=float)
control_ribbon = (
    ribbon_from_hill_params(controls, temps) if not controls.empty else pd.DataFrame()
)
eva_ribbon = ribbon_from_sweep_table(sweeps) if not sweeps.empty else pd.DataFrame()
ribbon = pd.concat(
    [df for df in (control_ribbon, eva_ribbon) if not df.empty], ignore_index=True
)

fig_ribbon = plot_ribbon_curves(ribbon, prototypes)
fig_violin = plot_violins(panel)
save_figure(fig_ribbon, paths["fig_dir"] / "ribbon_curves.png")
save_figure(fig_violin, paths["fig_dir"] / "cohort_violins.png")
plt.show()

tests = cohort_stat_tests(panel)
print("Stream vs Rfam / RefSeq / pilot")
display(tests.round(4) if not tests.empty else tests)

## §4 Multi-engine concordance (orthogonal consensus)

Compute Spearman $r_s$ and MAE between ViennaRNA and NUPACK. Flag software-fragile sequences ($|\Delta T_m| > 5^\circ\mathrm{C}$, or one engine predicts a switch while the other predicts an open coil).

The natural panel showed $r_s \approx 0.035$. Top-tier synthetic leads should keep $|\Delta T_m| \le 3^\circ\mathrm{C}$.


In [ ]:
fig_box = plot_nh_vs_tm(panel)
fig_leak = plot_leakiness(panel)
fig_parity = plot_engine_parity(panel)
fig_nov = plot_novelty_vs_nh(panel)
save_figure(fig_box, paths["fig_dir"] / "nh_vs_tm.png")
save_figure(fig_leak, paths["fig_dir"] / "leakiness_vs_delta_p.png")
save_figure(fig_parity, paths["fig_dir"] / "vienna_nupack_parity.png")
save_figure(fig_nov, paths["fig_dir"] / "novelty_vs_nh.png")
plt.show()

conc = engine_concordance(panel)
fragile = flag_software_fragile(panel)
print("Concordance:", conc)
print("software-fragile rows:", int(fragile.sum()))

## §5 The 4-gate visual diagnostic checklist

1. **Sigmoidal snap:** $n_{\mathrm{H}} > 1.5$
2. **Heat-shock midpoint:** $T_{\mathrm{m}} \in [42, 45]^\circ\mathrm{C}$
3. **Dynamic amplitude:** $\Delta\theta \ge 0.50$
4. **Tight baseline repression:** $P_{\mathrm{open}}^{37^\circ\mathrm{C}} \le 0.20$

Across the full 2,396 natural/control panel, **0** sequences pass all four gates simultaneously. EVA counts require Hill fits (set `RUN_EVA_SWEEPS = True` or load a cache).


In [ ]:
checklist = checklist_by_cohort(panel)
if not checklist.empty:
    write_checklist_summary(checklist, paths["checklist_json"])
    print("Wrote", paths["checklist_json"])
display(checklist.round(4) if not checklist.empty else checklist)

## §6 Top lead selection & structural export

$$\mathrm{Score} = w_1 \cdot n_{\mathrm{H}} + w_2 \cdot \Delta P_{\mathrm{RBS}} - w_3 \cdot \lvert T_{\mathrm{m}} - 43.5\rvert - w_4 \cdot P_{\mathrm{open}}^{37^\circ\mathrm{C}} - w_5 \cdot \lvert \Delta T_{\mathrm{m}}^{\mathrm{Vienna-NP}}\rvert$$

Export path: `data/processed/leads/eva_top10_experimental_leads.fasta`. Restriction screens: *EcoRI*, *BamHI*, *XhoI*. MFE dot-bracket + `RNAplot` PS when Vienna is installed.


In [ ]:
print("weights:", SCORE_WEIGHTS)
leads = rank_leads(eva_fitted, n=10)
if not leads.empty and leads["sequence"].notna().any():
    lead_path = export_leads(
        leads, fasta_path=paths["leads_fasta"], fig_dir=paths["fig_dir"] / "leads"
    )
    print("Wrote", lead_path)
show_cols = [
    c
    for c in (
        "record_id",
        "cohort",
        "composite_score",
        "viennarna_hill_coeff",
        "viennarna_Tm",
        "viennarna_delta_P_RBS",
        "viennarna_P_open_RBS_37",
        "viennarna_mfe_zscore",
        "E_Rfam",
        "software_fragile",
    )
    if c in leads.columns
]
display(leads[show_cols].round(4) if not leads.empty else leads)